<a href="https://colab.research.google.com/github/joaocanaslopes/ML_project_share_rep/blob/main/ML_project_load_training_save_HS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementação em PyTorch: Configuração de Modelos e Treinamento

 Esta seção irá implementar o processo usando **PyTorch**, seguindo  para seleção de dispositivo, configuração de camadas de transfer learning e loop de treinamento.

### 1. Seleção Automática do Dispositivo

In [1]:
import torch

# 1. Selecionar o dispositivo mais rápido disponível
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

device = get_device()
print(f"Dispositivo selecionado: {device}")

Dispositivo selecionado: cuda


### 2. Preparação dos Dados para PyTorch

Como o PyTorch usa `Dataset` e `DataLoader` em vez de `ImageDataGenerator`, precisamos configurar as transformações e carregar os dados novamente.

### Download e Preparação do Dataset do Kaggle (usando opendatasets)

O dataset foi baixado e extraído automaticamente na célula anterior (`f3e87527`) usando a biblioteca `opendatasets`. As instruções de API key do Kaggle são tratadas por lá.

In [7]:
# A instalação da biblioteca Kaggle já é tratada pelo opendatasets na célula de download.
# Esta célula está vazia para fins de organização e para evitar duplicação.

In [5]:
import sys
!pip install opendatasets
import opendatasets as od

# The Kaggle dataset URL for PlantVillage
dataset_url = 'https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset'

# Download the dataset
# opendatasets will prompt for Kaggle username and key if not found in environment variables
od.download(dataset_url)
print("Dataset downloaded successfully.")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: rainozita
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset


100%|██████████| 2.70G/2.70G [00:26<00:00, 110MB/s]



Dataset downloaded successfully.


Agora, vamos baixar o dataset. O nome do dataset no Kaggle é `vipulgohel/new-plant-diseases-dataset`.

In [8]:
# O download do dataset já é tratado pelo opendatasets na célula de download.
# Esta célula está vazia para fins de organização e para evitar duplicação.

O dataset foi descompactado automaticamente pela biblioteca `opendatasets`.

In [9]:
# A descompactação do dataset já é tratada pelo opendatasets na célula de download.
# Esta célula está vazia para fins de organização e para evitar duplicação.

Com o dataset devidamente baixado e extraído, a célula de preparação dos dados para PyTorch (`fdcbeaf7`) deve ser executada novamente.

In [6]:
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# Reutilizar variáveis definidas anteriormente a partir da configuração do TensorFlow/Keras
img_height, img_width = 224, 224
batch_size = 32
# Caminho corrigido após a extração do opendatasets, considerando a pasta aninhada
dataset_path = './new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
train_dir = os.path.join(dataset_path, 'train')
valid_dir = os.path.join(dataset_path, 'valid')
num_classes = 38 # Assumindo que este é o número correto de classes

# 2. Configurar transformações para dados de treino e validação
train_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.RandomRotation(40),
    transforms.RandomResizedCrop(img_height, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Carregar datasets
print(f"Verificando o caminho de treinamento: {train_dir}")
print(f"O diretório de treinamento existe? {os.path.exists(train_dir)}")
print(f"O caminho de treinamento é um diretório? {os.path.isdir(train_dir)}")

print(f"Verificando o caminho de validação: {valid_dir}")
print(f"O diretório de validação existe? {os.path.exists(valid_dir)}")
print(f"O caminho de validação é um diretório? {os.path.isdir(valid_dir)}")

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(valid_dir, transform=val_transforms)

# Criar DataLoaders
# Definindo num_workers=0 para evitar FileNotFoundError em alguns ambientes, como o Colab, ao usar múltiplos processos para carregar dados.
# num_workers > 0 pode causar problemas de acesso a arquivos quando os workers são processos filhos.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# Obter o número de classes a partir do dataset (deve ser o mesmo que num_classes)
num_classes_pytorch = len(train_dataset.classes)
print(f"Número de classes detectado pelo PyTorch: {num_classes_pytorch}")

print(f"Total de imagens de treinamento (PyTorch): {len(train_dataset)}")
print(f"Total de imagens de validação (PyTorch): {len(val_dataset)}")

Verificando o caminho de treinamento: ./new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
O diretório de treinamento existe? True
O caminho de treinamento é um diretório? True
Verificando o caminho de validação: ./new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid
O diretório de validação existe? True
O caminho de validação é um diretório? True
Número de classes detectado pelo PyTorch: 38
Total de imagens de treinamento (PyTorch): 70295
Total de imagens de validação (PyTorch): 17572


### 3. Configuração do Modelo ResNet50 (PyTorch)

In [21]:
import torch.nn as nn

# Carregar o modelo ResNet50 pré-treinado
# Usando weights='DEFAULT' para obter os pesos pré-treinados recomendados
resnet_model_pt = torchvision.models.resnet50(weights='DEFAULT')

# 2. Congelar todas as camadas
for param in resnet_model_pt.parameters():
    param.requires_grad = False

# Descongelar apenas 'layer4' e 'fc'
# 'layer4' é o último bloco convolucional do ResNet50
for param in resnet_model_pt.layer4.parameters():
    param.requires_grad = True

# 'fc' é a camada totalmente conectada final
resnet_model_pt.fc = nn.Linear(resnet_model_pt.fc.in_features, num_classes_pytorch)
# A camada 'fc' recém-criada tem requires_grad=True por padrão

# Mover o modelo para o dispositivo selecionado
resnet_model_pt = resnet_model_pt.to(device)

# Imprimir o número de parâmetros treináveis
trainable_params_resnet = sum(p.numel() for p in resnet_model_pt.parameters() if p.requires_grad)
print(f"ResNet50: Número de parâmetros treináveis: {trainable_params_resnet}")

# Exemplo: Imprimir alguns parâmetros de layer4 para verificar se estão descongelados
# print(f"ResNet50 layer4[2].bn3.weight requires_grad: {resnet_model_pt.layer4[2].bn3.weight.requires_grad}")
# print(f"ResNet50 fc.weight requires_grad: {resnet_model_pt.fc.weight.requires_grad}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 151MB/s]


ResNet50: Número de parâmetros treináveis: 15042598


### 4. Configuração do Modelo MobileNetV2 (PyTorch)

In [22]:
# Carregar o modelo MobileNetV2 pré-treinado
# Usando weights='DEFAULT' para obter os pesos pré-treinados recomendados
mobilenet_model_pt = torchvision.models.mobilenet_v2(weights='DEFAULT')

# 2. Congelar todas as camadas
for param in mobilenet_model_pt.parameters():
    param.requires_grad = False

# Descongelar apenas o 'classifier'
# No MobileNetV2, o 'classifier' é uma Sequential que contém a última camada linear
for param in mobilenet_model_pt.classifier.parameters():
    param.requires_grad = True

# Substituir a camada final do classificador
# O classificador do MobileNetV2 geralmente tem uma camada Linear final
in_features = mobilenet_model_pt.classifier[-1].in_features
mobilenet_model_pt.classifier[-1] = nn.Linear(in_features, num_classes_pytorch)

# Mover o modelo para o dispositivo selecionado
mobilenet_model_pt = mobilenet_model_pt.to(device)

# Imprimir o número de parâmetros treináveis
trainable_params_mobilenet = sum(p.numel() for p in mobilenet_model_pt.parameters() if p.requires_grad)
print(f"MobileNetV2: Número de parâmetros treináveis: {trainable_params_mobilenet}")

# Exemplo: Imprimir alguns parâmetros do classifier para verificar se estão descongelados
# print(f"MobileNetV2 classifier[1].weight requires_grad: {mobilenet_model_pt.classifier[1].weight.requires_grad}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 103MB/s] 

MobileNetV2: Número de parâmetros treináveis: 48678


### 5. Loop de Treinamento e Validação (PyTorch)

Vamos criar uma função genérica de treinamento que pode ser usada para ambos os modelos, incorporando as movimentações para o dispositivo e o cálculo de métricas.

In [28]:
import time
import copy
import torch
import torch.optim as optim

def train_model_pytorch(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs,
    device,
    patience=2  # early stopping
):

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    counter = 0

    print(f"Iniciando treinamento no dispositivo: {device}")
    print(f"Número de épocas: {num_epochs}")

    for epoch in range(num_epochs):

        start_time = time.time()

        # =========================
        # TREINO
        # =========================
        model.train()

        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_train / total_train

        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)

        # =========================
        # VALIDAÇÃO
        # =========================
        model.eval()

        running_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_val / total_val

        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        # =========================
        # LOG
        # =========================
        end_time = time.time()
        epoch_duration = end_time - start_time

        print(f"Epoch {epoch+1}/{num_epochs} - Tempo: {epoch_duration:.2f}s | "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

        # =========================
        # SAVE BEST MODEL + EARLY STOPPING
        # =========================
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

            torch.save(
                model.state_dict(),
                f'best_{model.__class__.__name__}_pytorch.pth'
            )

            print("✔ Melhor modelo salvo")
            counter = 0

        else:
            counter += 1
            print(f"EarlyStopping: {counter}/{patience}")

        if counter >= patience:
            print("⛔ Early stopping ativado!")
            break

    # carregar melhor modelo
    model.load_state_dict(best_model_wts)

    return history

### 6. Treinamento do ResNet50 (PyTorch)

Sugestão de hiperparâmetros:
*   **Batch Size**: `32` (já definido)
*   **Número de Épocas**: `10-20` (começaremos com `10` para demonstração)
*   **Learning Rate**: `1e-4` (um valor comum para fine-tuning)

In [31]:
import torch.optim as optim

num_epochs_pt = 4 #
learning_rate_pt = 1e-4 # Sugestão: 1e-4 para fine-tuning

# Definir função de perda e otimizador para ResNet50
criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model_pt.parameters()),
    lr=learning_rate_pt
)

print("\n--- Treinando ResNet50 (PyTorch) ---")
history_resnet_pt = train_model_pytorch(
    resnet_model_pt,
    train_loader,
    val_loader,
    criterion_resnet,
    optimizer_resnet,
    num_epochs_pt,
    device
)


--- Treinando ResNet50 (PyTorch) ---
Iniciando treinamento no dispositivo: cuda
Número de épocas: 4
Epoch 1/4 - Tempo: 763.13s | Train Loss: 0.0245 | Train Acc: 0.9920 | Val Loss: 0.0150 | Val Acc: 0.9957
✔ Melhor modelo salvo
Epoch 2/4 - Tempo: 773.01s | Train Loss: 0.0177 | Train Acc: 0.9943 | Val Loss: 0.0176 | Val Acc: 0.9943
EarlyStopping: 1/2
Epoch 3/4 - Tempo: 726.68s | Train Loss: 0.0162 | Train Acc: 0.9947 | Val Loss: 0.0150 | Val Acc: 0.9956
EarlyStopping: 2/2
⛔ Early stopping ativado!


### 7. Treinamento do MobileNetV2 (PyTorch)

Usaremos os mesmos hiperparâmetros sugeridos para o MobileNetV2.

In [32]:
import torch.optim as optim

# Definir função de perda e otimizador para MobileNetV2
criterion_mobilenet = nn.CrossEntropyLoss()
optimizer_mobilenet = optim.Adam(
    filter(lambda p: p.requires_grad, mobilenet_model_pt.parameters()),
    lr=learning_rate_pt
)

print("\n--- Treinando MobileNetV2 (PyTorch) ---")
history_mobilenet_pt = train_model_pytorch(
    mobilenet_model_pt,
    train_loader,
    val_loader,
    criterion_mobilenet,
    optimizer_mobilenet,
    num_epochs_pt,
    device
)


--- Treinando MobileNetV2 (PyTorch) ---
Iniciando treinamento no dispositivo: cuda
Número de épocas: 4
Epoch 1/4 - Tempo: 481.28s | Train Loss: 0.8496 | Train Acc: 0.8537 | Val Loss: 0.6864 | Val Acc: 0.8674
✔ Melhor modelo salvo
Epoch 2/4 - Tempo: 466.20s | Train Loss: 0.5933 | Train Acc: 0.8770 | Val Loss: 0.5189 | Val Acc: 0.8860
✔ Melhor modelo salvo
Epoch 3/4 - Tempo: 477.70s | Train Loss: 0.4826 | Train Acc: 0.8919 | Val Loss: 0.4308 | Val Acc: 0.9002
✔ Melhor modelo salvo
Epoch 4/4 - Tempo: 482.17s | Train Loss: 0.4193 | Train Acc: 0.9016 | Val Loss: 0.3836 | Val Acc: 0.9087
✔ Melhor modelo salvo


In [33]:
import os
import torch

# Definir os nomes dos arquivos para salvar os modelos PyTorch
resnet_save_path_pt = 'final_resnet_model_pytorch.pth'
mobilenet_save_path_pt = 'final_mobilenet_model_pytorch.pth'

# Verificar se os modelos PyTorch existem (objetos em memória) antes de tentar salvá-los
# 'resnet_model_pt' e 'mobilenet_model_pt' são definidos nas células de configuração do modelo

if 'resnet_model_pt' in locals() and resnet_model_pt is not None:
    try:
        # Salva apenas o estado do modelo (pesos e vieses)
        torch.save(resnet_model_pt.state_dict(), resnet_save_path_pt)
        print(f"Modelo ResNet50 PyTorch salvo em: {resnet_save_path_pt}")
    except Exception as e:
        print(f"Erro ao salvar o modelo ResNet50 PyTorch: {e}")
else:
    print("O modelo ResNet50 PyTorch não foi encontrado ou não foi treinado.")

if 'mobilenet_model_pt' in locals() and mobilenet_model_pt is not None:
    try:
        # Salva apenas o estado do modelo (pesos e vieses)
        torch.save(mobilenet_model_pt.state_dict(), mobilenet_save_path_pt)
        print(f"Modelo MobileNetV2 PyTorch salvo em: {mobilenet_save_path_pt}")
    except Exception as e:
        print(f"Erro ao salvar o modelo MobileNetV2 PyTorch: {e}")
else:
    print("O modelo MobileNetV2 PyTorch não foi encontrado ou não foi treinado.")

print("Os callbacks de treinamento na função 'train_model_pytorch' já salvaram os melhores modelos como 'best_ResNet_pytorch.pth' e 'best_MobileNetV2_pytorch.pth' durante o treinamento.")

Modelo ResNet50 PyTorch salvo em: final_resnet_model_pytorch.pth
Modelo MobileNetV2 PyTorch salvo em: final_mobilenet_model_pytorch.pth
Os callbacks de treinamento na função 'train_model_pytorch' já salvaram os melhores modelos como 'best_ResNet_pytorch.pth' e 'best_MobileNetV2_pytorch.pth' durante o treinamento.


## Deploy no Hugging Face Spaces

Para verificar se a aplicação consegue identificar as doenças, podemos criar uma interface simples com Gradio e implantá-la no Hugging Face Spaces.

Primeiro, certifique-se de que o treinamento de ambos os modelos (ResNet50 e MobileNetV2) foi concluído e os arquivos `best_resnet_model.h5` e `best_mobilenet_model.h5` foram salvos. Estes serão os pesos que carregaremos para a inferência.

### Passos para criar a aplicação Gradio e implantar:

1.  **Instalar Gradio**
2.  **Carregar os modelos**
3.  **Definir uma função de previsão**
4.  **Criar a interface Gradio**
5.  **Preparar para o Hugging Face Spaces**

In [34]:
# Instalar a biblioteca Gradio, se ainda não estiver instalada
!pip install gradio

### Carregar Modelos e Definir Função de Previsão

Vamos carregar os modelos salvos e definir uma função que receberá uma imagem, fará o pré-processamento necessário e retornará a previsão do modelo.

In [35]:
import torch
import torchvision
from torchvision import transforms
import numpy as np
from PIL import Image
import gradio as gr
import os

# --- Modelos PyTorch ---
# Carregar os modelos PyTorch treinados
# Certifique-se de que os arquivos .pth existem após o treinamento

# Re-inicializar os modelos com as arquiteturas corretas e carregar os pesos
# ResNet50
# É necessário ter num_classes e device definidos, que vêm de células anteriores

resnet_path_pt = 'best_ResNet_pytorch.pth' # Nome usado na função de treinamento
resnet_model_pt_inf = None

# As variáveis num_classes_pytorch e device são globais e devem estar acessíveis
if os.path.exists(resnet_path_pt):
    resnet_model_pt_inf = torchvision.models.resnet50(weights=None) # Inicia sem pesos pré-treinados
    # Substitui a camada de classificação final
    resnet_model_pt_inf.fc = torch.nn.Linear(resnet_model_pt_inf.fc.in_features, num_classes_pytorch)
    try:
        resnet_model_pt_inf.load_state_dict(torch.load(resnet_path_pt, map_location=device))
        resnet_model_pt_inf.to(device)
        resnet_model_pt_inf.eval() # Define o modo de avaliação
        print(f"Modelo ResNet50 PyTorch carregado de {resnet_path_pt}")
    except Exception as e:
        print(f"Erro ao carregar o modelo ResNet50 PyTorch: {e}")
        resnet_model_pt_inf = None
else:
    print(f"Erro: Modelo ResNet50 PyTorch não encontrado em {resnet_path_pt}.")

# MobileNetV2
mobilenet_path_pt = 'best_MobileNetV2_pytorch.pth' # Nome usado na função de treinamento
mobilenet_model_pt_inf = None

# As variáveis num_classes_pytorch e device são globais e devem estar acessíveis
if os.path.exists(mobilenet_path_pt):
    mobilenet_model_pt_inf = torchvision.models.mobilenet_v2(weights=None) # Inicia sem pesos pré-treinados
    # Substitui a camada de classificação final
    in_features_mobilenet = mobilenet_model_pt_inf.classifier[-1].in_features
    mobilenet_model_pt_inf.classifier[-1] = torch.nn.Linear(in_features_mobilenet, num_classes_pytorch)
    try:
        mobilenet_model_pt_inf.load_state_dict(torch.load(mobilenet_path_pt, map_location=device))
        mobilenet_model_pt_inf.to(device)
        mobilenet_model_pt_inf.eval() # Define o modo de avaliação
        print(f"Modelo MobileNetV2 PyTorch carregado de {mobilenet_path_pt}")
    except Exception as e:
        print(f"Erro ao carregar o modelo MobileNetV2 PyTorch: {e}")
        mobilenet_model_pt_inf = None
else:
    print(f"Erro: Modelo MobileNetV2 PyTorch não encontrado em {mobilenet_path_pt}.")

# Recuperar os nomes das classes do train_dataset (PyTorch)
# As classes foram carregadas quando train_dataset foi criado em fdcbeaf7
# Assume que 'train_dataset' está disponível do escopo global
class_names = sorted(train_dataset.classes) if 'train_dataset' in locals() else []

# Define uma transformação para inferência (correspondendo às transformações de validação)
inference_transforms = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_image_pytorch(image, model_choice):
    if image is None:
        return "Por favor, carregue uma imagem.", {}

    # Converte imagem PIL para Tensor PyTorch e aplica transformações
    img = Image.fromarray(image.astype('uint8'), 'RGB')
    img_tensor = inference_transforms(img).unsqueeze(0) # Adiciona dimensão do batch
    img_tensor = img_tensor.to(device)

    model_to_use = None
    if model_choice == "ResNet50" and resnet_model_pt_inf:
        model_to_use = resnet_model_pt_inf
    elif model_choice == "MobileNetV2" and mobilenet_model_pt_inf:
        model_to_use = mobilenet_model_pt_inf
    else:
        return f"Erro: Modelo {model_choice} não disponível ou não carregado.", {}

    with torch.no_grad():
        outputs = model_to_use(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
        predicted_class_idx = torch.argmax(probabilities).item()
        predicted_class_name = class_names[predicted_class_idx]
        confidence = probabilities[predicted_class_idx].item()

    # Prepara resultados para exibição no Gradio
    prediction_output = {
        class_names[i]: float(probabilities[i]) for i in range(len(class_names))
    }

    return f"Classe Prevista: {predicted_class_name} (Confiança: {confidence:.2f})", prediction_output

Modelo ResNet50 PyTorch carregado de best_ResNet_pytorch.pth
Modelo MobileNetV2 PyTorch carregado de best_MobileNetV2_pytorch.pth


### Criar e Lançar a Interface Gradio

Agora vamos usar o Gradio para criar uma interface interativa. Você pode executar este código no Colab e ele fornecerá um link público para testar a interface diretamente. Para implantar no Hugging Face Spaces, você precisará salvar esses arquivos em uma estrutura específica.

In [36]:
# Criar a interface Gradio
# Verifica se pelo menos um modelo PyTorch foi carregado com sucesso
if ('resnet_model_pt_inf' in locals() and resnet_model_pt_inf is not None) or \
   ('mobilenet_model_pt_inf' in locals() and mobilenet_model_pt_inf is not None):
    demo = gr.Interface(
        fn=predict_image_pytorch, # Usa a função de previsão PyTorch
        inputs=[
            gr.Image(type="numpy", label="Upload de Imagem de Folha de Planta"),
            gr.Radio([m for m, model_obj in {"ResNet50": resnet_model_pt_inf, "MobileNetV2": mobilenet_model_pt_inf}.items() if model_obj is not None],
                     label="Escolha do Modelo",
                     value="ResNet50" if resnet_model_pt_inf else "MobileNetV2")
        ],
        outputs=[
            gr.Label(label="Resultado da Previsão"),
            gr.Label(label="Probabilidades")
        ],
        title="Classificação de Doenças de Plantas (PyTorch)",
        description="Faça upload de uma imagem de folha de planta para prever a doença usando modelos de Transfer Learning (ResNet50 ou MobileNetV2) em PyTorch."
    )

    # Lançar a interface (isso criará um link público temporário no Colab)
    demo.launch()
else:
    print("Não é possível lançar a interface Gradio: nenhum modelo PyTorch foi carregado com sucesso.")

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://377c33c14316a18fb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Como implantar no Hugging Face Spaces:

1.  **Crie uma nova Space** no Hugging Face (huggingface.co/spaces).
2.  Escolha **Gradio** como o SDK e **Docker** como o tipo de ambiente.
3.  **Crie um arquivo `app.py`** no seu repositório do Hugging Face Spaces e copie o código da célula acima (`gr.Interface(...)` e `demo.launch(...)`) para ele. Certifique-se de incluir todos os `imports` necessários (`tensorflow`, `numpy`, `PIL`, `gradio`, `os`).
4.  **Crie um arquivo `requirements.txt`** com as seguintes linhas:
    ```
    tensorflow
    numpy
    Pillow
    gradio
    ```
5.  **Faça upload dos seus modelos treinados** (`best_resnet_model.h5` e `best_mobilenet_model.h5`) para o mesmo diretório no seu repositório do Hugging Face Spaces. Você pode arrastá-los e solá-los ou usar `git LFS` para arquivos grandes.
6.  **Certifique-se de que os `class_names` estejam definidos** no `app.py`. Você pode copiá-los do seu notebook (após o `train_generator` ser criado) ou salvá-los em um arquivo separado e carregá-los.
    ```python
    # Exemplo de como você pode definir class_names em app.py
    class_names = [
        'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
        'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
        'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
        'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot',
        'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
        'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
        'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
        'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy',
        'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy',
        'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
        'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
        'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
        'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy'
    ]
    ```
7.  O Hugging Face Spaces detectará o `app.py` e os `requirements.txt` e construirá seu Space automaticamente.